03 · Scoring: is the agent right, and does the follow-up loop help?

Reads the run folders written by `02_agent_adk.ipynb` and scores them four ways:

| Part | Question | Answer key |
|---|---|---|
| 1. Labels | Do our labelers agree? (Cohen's κ) | the label sheets themselves |
| 2. Track A | Did the agent find the right clause? | CUAD lawyer highlights (`cuad_gold_spans.csv`) |
| 3. Track B | Is the verdict right? False alarms and missed risks per clause type | our gold labels |
| 4. Loop | When the follow-up loop changed a verdict, did it fix it or break it? What did it cost? | gold labels + traces |
| 5. Track C | Are the revision suggestions useful? | our 1–3 ratings |

Every table is also saved to `scores/` so the report and slides use the same numbers.

## 0. Setup

Run this in a new Colab notebook. When it asks, upload the `runs_test.zip` (or `runs_dev.zip`) that notebook 02 downloaded; it holds `runs/{split}_rag`, `runs/{split}_single_pass` and `runs/{split}_adaptive`.

In [ ]:
import os, sys, json, zipfile
from pathlib import Path
import pandas as pd
from sklearn.metrics import cohen_kappa_score

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata, files
    if not Path("contract-review-agent").exists():
        token = userdata.get("GH_TOKEN")
        BRANCH = "main"
        !git clone -q -b {BRANCH} https://{token}@github.com/JJJJeny/contract-review-agent.git
    DATA_DIR = Path("contract-review-agent")
    for name in files.upload():                  # upload one or more run.zip files (optional)
        zipfile.ZipFile(name).extractall(".")
else:
    DATA_DIR = Path(".")

SPLIT = "test"                                   # "dev" while tuning, "test" for the final numbers
CONDITIONS = ["rag", "single_pass", "adaptive"]  # baseline -> loop off -> full agent
FAMILIES = ["payment", "termination", "liability_indemnity", "confidentiality", "renewal", "dispute_governing_law"]
OUT = Path("scores"); OUT.mkdir(exist_ok=True)


def find_run(cond):
    for root in (Path("."), Path("runs"), DATA_DIR / "runs"):
        if (root / f"{SPLIT}_{cond}" / "results.csv").exists():
            return root / f"{SPLIT}_{cond}"


RUNS = {c: find_run(c) for c in CONDITIONS}
RUNS = {c: d for c, d in RUNS.items() if d}
print("found runs:", {c: str(d) for c, d in RUNS.items()})
missing = set(CONDITIONS) - set(RUNS)
if missing:
    print("not found (skipped below):", sorted(missing))

pred = pd.concat([pd.read_csv(d / "results.csv").assign(condition=c) for c, d in RUNS.items()], ignore_index=True)


def save(df, name):
    df.to_csv(OUT / f"{name}.csv")
    return df

## 1. Labels and agreement (Cohen's κ)

Each label sheet has one row per contract × family. κ is agreement corrected for chance: (observed − chance) / (1 − chance).
Verdicts have no order, so plain κ. Severities do (n/a < low < medium < high), so weighted κ: being one level off costs less than being two levels off.

How our labels were made (say this in the report): each of us started from an AI-drafted sheet and reviewed it row by row against the contract and the rubric. The two reviewed sheets differ on 8 of 90 rows; those 8 were adjudicated against the contract text and the rubric, with the reason written in the `adjudication` column of `labels_gold.xlsx`. Gold is the answer key for everything below.

In [ ]:
SHEETS = {"jenny": "Jenny_Labels_claude.xlsx", "kayla": "Kayla's Labels.xlsx", "gold": "labels_gold.xlsx"}
SEV = {"n/a": 0, "low": 1, "medium": 2, "high": 3}


def load_labels(path):
    df = pd.read_excel(path).dropna(subset=["contract_id", "family"])
    df = df[df.family.isin(FAMILIES)].copy()
    df["verdict"] = df.verdict.str.strip().str.lower()
    df["severity"] = df.severity.fillna("n/a").astype(str).str.strip().str.lower().replace({"nan": "n/a"})
    return df.set_index(["contract_id", "family"])


labels = {who: load_labels(DATA_DIR / f) for who, f in SHEETS.items()}
gold = labels["gold"][["verdict", "severity"]]

pair = labels["jenny"][["verdict", "severity"]].join(labels["kayla"][["verdict", "severity"]], lsuffix="_j", rsuffix="_k")
by = {"all": pair, **{f: pair.xs(f, level="family") for f in FAMILIES}}
display(save(pd.DataFrame({
    k: {"rows": len(g), "verdict_agree": (g.verdict_j == g.verdict_k).mean(),
        "verdict_kappa": cohen_kappa_score(g.verdict_j, g.verdict_k),
        "severity_wkappa": cohen_kappa_score(g.severity_j.map(SEV), g.severity_k.map(SEV), weights="quadratic")}
    for k, g in by.items()}).T.round(2), "kappa_jenny_vs_kayla"))
display(pd.crosstab(pair.verdict_j.rename("jenny"), pair.verdict_k.rename("kayla")))

# rows the two of us disagreed on: used in part 4 to see whether the agent asks a human where humans found it hard
hard = set(pair[(pair.verdict_j != pair.verdict_k) | (pair.severity_j != pair.severity_k)].index)
display(labels["gold"].loc[sorted(hard), ["verdict", "severity", "source", "adjudication"]])

## 2. Track A: clause detection against CUAD

CUAD only covers four of our families, and only part of each (see the README). For those, a lawyer highlighted the clause text. We count the agent's quote as a **hit** if it overlaps any CUAD highlight for that contract and family.

* **precision** = hits / rows where the agent quoted something
* **recall** = hits / rows where CUAD has a highlight
* **char F1** = how much of the highlighted text the quotes cover, character by character (a stricter, partial-credit measure)

A quote with no CUAD highlight is not always wrong: CUAD's categories are narrower than ours (e.g. termination for cause is not a CUAD category). So precision here is a lower bound.

In [ ]:
cuad = pd.read_csv(DATA_DIR / "cuad_gold_spans.csv")
cuad["span"] = list(zip(cuad.answer_start, cuad.answer_start + cuad.text.str.len()))
cuad_spans = cuad.groupby(["contract_id", "family"]).span.apply(list)
CUAD_FAMILIES = sorted(cuad.family.unique())


def chars(spans):
    return set().union(*(range(s, e) for s, e in spans)) if spans else set()


def track_a(df):
    rows = []
    for r in df[df.family.isin(CUAD_FAMILIES)].itertuples():
        got = [tuple(s) for s in json.loads(r.evidence_spans or "[]")]
        want = cuad_spans.get((r.contract_id, r.family), [])
        g, w = chars(got), chars(want)
        rows.append({"family": r.family, "quoted": bool(got), "has_cuad": bool(want),
                     "hit": bool(g & w), "char_tp": len(g & w), "char_pred": len(g), "char_gold": len(w)})
    t = pd.DataFrame(rows)

    def score(g):
        p = g.hit.sum() / max(g.quoted.sum(), 1)
        r = g.hit.sum() / max(g.has_cuad.sum(), 1)
        cp, cr = g.char_tp.sum() / max(g.char_pred.sum(), 1), g.char_tp.sum() / max(g.char_gold.sum(), 1)
        return pd.Series({"n": len(g), "precision": p, "recall": r, "f1": 2 * p * r / max(p + r, 1e-9),
                          "char_f1": 2 * cp * cr / max(cp + cr, 1e-9)})
    return pd.concat([t.groupby("family").apply(score), score(t).to_frame("all").T]).round(2)


track_a_tables = {c: track_a(pred[pred.condition == c]) for c in RUNS}
display(save(pd.concat(track_a_tables, names=["condition"]), "track_a_clause_detection"))

## 3. Track B: risk verdicts against our gold labels

From the small-business owner's side, the two mistakes that matter are:

* **false alarm**: the agent flags a clause (risky or missing) that gold says is acceptable. The owner wastes a negotiation point.
* **missed risk**: gold says risky or missing, and the agent says acceptable. The owner signs something bad. This is the worse mistake.

`unclear` is neither: the agent declined to decide. We report how often it does that, and whether it does it on rows gold also calls unclear.

In [ ]:
FLAG = {"risky", "missing"}


def rate(hits, total):
    return hits.sum() / total.sum() if total.sum() else float("nan")   # NaN, not 0, when nothing to count


def track_b(df):
    m = df.set_index(["contract_id", "family"])[["verdict"]].join(gold.verdict.rename("gold"), how="inner")

    def score(g):
        ok, flag = g.gold == "acceptable", g.gold.isin(FLAG)
        return pd.Series({
            "n": len(g),
            "accuracy": (g.verdict == g.gold).mean(),
            "false_alarm_rate": rate(ok & g.verdict.isin(FLAG), ok),
            "missed_risk_rate": rate(flag & (g.verdict == "acceptable"), flag),
            "missing_recall": rate((g.gold == "missing") & (g.verdict == "missing"), g.gold == "missing"),
            "said_unclear": (g.verdict == "unclear").mean(),
            "unclear_when_gold_unclear": rate((g.verdict == "unclear") & (g.gold == "unclear"), g.verdict == "unclear"),
        })
    by_fam = m.groupby(level="family").apply(score)
    return pd.concat([by_fam, score(m).to_frame("all").T]).round(2), m


track_b_tables, joined = {}, {}
for c in RUNS:
    track_b_tables[c], joined[c] = track_b(pred[pred.condition == c])
display(save(pd.concat(track_b_tables, names=["condition"]), "track_b_verdicts"))

The headline comparison, one row per condition. If the loop is worth its cost, `adaptive` should beat `single_pass` on missed risks without raising false alarms.

In [ ]:
headline = pd.DataFrame({c: t.loc["all"] for c, t in track_b_tables.items()}).T
for c in RUNS:
    s = pd.DataFrame(json.load(open(RUNS[c] / "summary.json"))["contracts"])
    headline.loc[c, "cost_usd"] = s.cost_usd.sum()
    headline.loc[c, "minutes"] = s.elapsed_s.sum() / 60
    headline.loc[c, "tool_calls"] = s.tool_calls.sum()
display(save(headline.round(3), "headline"))

ax = pd.DataFrame({c: t.drop("all")[["false_alarm_rate", "missed_risk_rate"]].stack() for c, t in track_b_tables.items()}) \
       .unstack().loc[:, (slice(None), "missed_risk_rate")].droplevel(1, axis=1) \
       .plot.bar(figsize=(8, 3.5), title="Missed risk rate by clause type (lower is better)", rot=20)
ax.figure.tight_layout(); ax.figure.savefig(OUT / "missed_risk_by_family.png", dpi=150)

## 4. Did the follow-up loop help?

In the adaptive run every row keeps the pre-pass verdict (`prepass_verdict`) next to the final one. For the rows the loop changed, compare both to gold:

* **fixed** wrong → right, **broke** right → wrong, **moved** wrong → a different wrong.

In [ ]:
if "adaptive" in RUNS:
    a = pred[pred.condition == "adaptive"].set_index(["contract_id", "family"]).join(gold.verdict.rename("gold"), how="inner")
    looped = a[a.path != "prepass"]
    changed = looped[looped.verdict_changed.astype(str).str.lower() == "true"]
    before_ok, after_ok = changed.prepass_verdict == changed.gold, changed.verdict == changed.gold
    flips = pd.Series({
        "rows": len(a), "entered_loop": len(looped), "verdict_changed": len(changed),
        "fixed (wrong->right)": int((~before_ok & after_ok).sum()),
        "broke (right->wrong)": int((before_ok & ~after_ok).sum()),
        "moved (wrong->wrong)": int((~before_ok & ~after_ok).sum()),
        "prepass_accuracy": (a.prepass_verdict == a.gold).mean().round(2),
        "final_accuracy": (a.verdict == a.gold).mean().round(2),
    })
    display(save(flips.to_frame("count"), "loop_flips"))
    display(changed[["prepass_verdict", "verdict", "gold", "triggers_fired", "steps"]])

### Trajectories: what the loop did

Does the loop stop on its own? Every looped family ends one of two ways: the model calls `finalize`, or a bound stops it (step limit, the same call repeated, no new evidence for two steps, tool-call cap, budget). A healthy agent finalizes most families itself; forced stops are the "looping" we want to see go down. Then: actions per family, and which rows went to the human queue. Rows the labelers disagreed on (`hard`) are where we would want the agent to ask a human.

In [ ]:
if "adaptive" in RUNS:
    tr = pd.read_json(RUNS["adaptive"] / "trace.jsonl", lines=True)
    ended = looped.path.str.replace("forced:", "stopped: ", regex=False).replace({"loop": "model called finalize"})
    display(save(ended.value_counts().to_frame("families"), "how_loops_ended"))
    loop_steps = tr[tr.phase.eq("loop")] if "phase" in tr else tr
    display(save(loop_steps.action.value_counts().to_frame("calls"), "action_mix"))
    print("steps per looped family:", looped.steps.describe()[["mean", "50%", "max"]].round(2).to_dict())

    q = pd.DataFrame(json.load(open(RUNS["adaptive"] / "human_queue.json")))
    if len(q):
        q["kind"] = q.question.str.extract(r"Agent stopped \(([^:)]+)")[0].fillna("agent asked")
        q["hard_row"] = [(r.contract_id, r.family) in hard for r in q.itertuples()]
        display(save(q.groupby("kind").agg(n=("family", "size"), on_hard_rows=("hard_row", "sum")), "human_queue"))
    asked = {(r.contract_id, r.family) for r in q.itertuples()} if len(q) else set()
    hard_in_run = hard & set(a.index)
    print(f"hard rows sent to a human: {len(hard_in_run & asked)} / {len(hard_in_run)};",
          f"easy rows sent to a human: {len(asked - hard)} / {len(set(a.index) - hard)}")

## 5. Graders: deterministic checks and an LLM judge

Two kinds of graders score every finding without a human:

* **Deterministic** (free, exact, run on all three conditions): is the output well formed, and does it follow the rules we can check in code?
* **LLM as a judge** (costs API calls): does the quoted evidence actually support the verdict under the rubric, and is the revision ask useful? A judge is only worth trusting if it agrees with people, so we check it against the gold labels and against our own Track C ratings.

In [ ]:
VERDICTS, SEVERITIES = {"acceptable", "risky", "missing", "unclear"}, {"high", "medium", "low", "n/a"}


def deterministic_checks(df, plans=None):
    quoted = df.evidence_text.fillna("").str.strip().replace("NONE FOUND", "") != ""
    checks = pd.DataFrame({
        "valid_verdict": df.verdict.isin(VERDICTS),
        "valid_severity": df.severity.fillna("n/a").isin(SEVERITIES),
        "severity_matches_verdict": (df.verdict == "acceptable") == (df.severity.fillna("n/a") == "n/a"),
        "quote_found_in_contract": df.evidence_verified.astype(str).str.lower() == "true",
        "flag_has_a_quote": ~df.verdict.isin({"risky", "unclear"}) | quoted,
        "missing_has_search_record": (df.verdict != "missing") | df.notes.fillna("").str.contains("search", case=False),
    })
    out = checks.mean()
    if plans is not None:                 # revision plan sorted by rubric §7: severity, then verdict
        sev, ver = {"high": 0, "medium": 1, "low": 2}, {"risky": 0, "missing": 1, "unclear": 2}
        keys = [[(sev.get(i.get("severity"), 3), ver.get(i.get("verdict"), 3)) for i in items] for items in plans.values()]
        out["plan_in_rubric_order"] = sum(k == sorted(k) for k in keys) / max(len(keys), 1)
    return out


det = pd.DataFrame({c: deterministic_checks(pred[pred.condition == c],
                                            json.load(open(RUNS[c] / "revision_plans.json")))
                    for c in RUNS}).round(2)
display(save(det, "deterministic_checks"))

### LLM as a judge

The judge sees the rubric for the family, the agent's verdict, its quote and its notes, and answers two questions on a 1–3 scale. It never sees the gold label. Set `JUDGE_MODEL` to a stronger model than the agent's if the course credit allows, so the judge is not grading its own mistakes. `MAX_JUDGE_CALLS` caps the cost.

In [ ]:
import re
from openai import OpenAI

JUDGE_MODEL = "@30875-fall26-openai/gpt-5-nano"     # a stronger model is better if available
MAX_JUDGE_CALLS = 300
api_key = userdata.get("OPENAI_API_KEY") if IN_COLAB else os.environ.get("OPENAI_API_KEY")
judge_client = OpenAI(api_key=api_key, base_url="https://api.portkey.ai/v1") if api_key else None

rubric_text = (DATA_DIR / "rubric_v1.md").read_text()
RUBRIC_BY_FAMILY = {f: re.search(rf"^## \d+\. {f}\n(.*?)(?=^## )", rubric_text, re.S | re.M).group(1) for f in FAMILIES}

FINDING_JUDGE = """You grade a contract reviewer's finding for a small-business customer, using this rubric:

{rubric}

Finding: family={family}, verdict={verdict}, severity={severity}
Quoted evidence ({section}): "{quote}"
Reviewer's notes: {notes}

Answer in JSON only: {{"supported": 1-3, "answers_rubric": 1-3, "reason": "one sentence"}}
supported: 3 = the quote clearly justifies this verdict and severity under the rubric, 2 = partly, 1 = it does not (wrong verdict, wrong severity, or the quote is about something else).
answers_rubric: 3 = the notes address the rubric sub-criterion that decides this family, 1 = they do not."""

PLAN_JUDGE = """A small-business customer will send this ask to a vendor about a contract clause.
Finding: {family} ({verdict}, {severity}). Clause: "{quote}"
Ask: {ask}
Fallback: {fallback}

Answer in JSON only: {{"grounded": 1-3, "specific": 1-3, "actionable": 1-3, "reason": "one sentence"}}
grounded: follows from the clause. specific: says exactly what to change (a number, a period, a cap). actionable: could be sent as is."""

calls = 0


def judge(prompt):
    global calls
    if judge_client is None or calls >= MAX_JUDGE_CALLS:
        return {}
    calls += 1
    reply = judge_client.chat.completions.create(model=JUDGE_MODEL, messages=[{"role": "user", "content": prompt}])
    text = reply.choices[0].message.content or ""
    try:
        return json.loads(text[text.index("{"):text.rindex("}") + 1])
    except ValueError:
        return {}


def judge_findings(df):
    out = []
    for r in df.itertuples():
        j = judge(FINDING_JUDGE.format(rubric=RUBRIC_BY_FAMILY[r.family], family=r.family, verdict=r.verdict,
                                       severity=r.severity, section=r.section_ref, quote=str(r.evidence_text)[:800],
                                       notes=str(r.notes)[:600]))
        out.append({"contract_id": r.contract_id, "family": r.family, **j})
    return pd.DataFrame(out).set_index(["contract_id", "family"])


JUDGED = {}
for c in RUNS:
    JUDGED[c] = judge_findings(pred[pred.condition == c]).join(gold.verdict.rename("gold")) \
        .join(pred[pred.condition == c].set_index(["contract_id", "family"]).verdict)
print(calls, "judge calls")

Is the judge trustworthy? If it is, findings whose verdict matches gold should get high `supported` scores and wrong ones low. `judge_separates` is the mean score on right findings minus the mean on wrong ones (bigger is better; near 0 means the judge cannot tell them apart).

In [ ]:
rows = {}
for c, j in JUDGED.items():
    if "supported" not in j:
        continue
    right = j.verdict == j.gold
    rows[c] = {"mean_supported": j.supported.mean(), "mean_answers_rubric": j.answers_rubric.mean(),
               "supported_when_right": j.supported[right].mean(), "supported_when_wrong": j.supported[~right].mean(),
               "judge_separates": j.supported[right].mean() - j.supported[~right].mean(),
               "judge_flags_wrong (score 1)": (j.supported[~right] == 1).mean()}
if rows:
    display(save(pd.DataFrame(rows).T.round(2), "llm_judge_findings"))

## 6. Track C: are the revision suggestions useful?

Each of us rates every revision-plan item from 1 to 3 on three questions, without looking at the other's sheet:

* **grounded**: does it follow from the quoted clause?
* **specific**: does it say what to change (a number, a notice period, a cap)?
* **actionable**: could the owner send it to the vendor as is?

Run the next cell once to write the rating sheet and let the LLM judge rate the same items. Fill in the sheet (one copy each, e.g. `ratings_jenny.csv`, `ratings_kayla.csv`), then run the cell after: it compares us with each other and with the judge.

In [ ]:
if "adaptive" in RUNS:
    plans = json.load(open(RUNS["adaptive"] / "revision_plans.json"))
    sheet = pd.DataFrame([{"contract_id": cid, "id": it["id"], "family": it["family"], "severity": it.get("severity"),
                           "evidence": it.get("evidence_text", "")[:300], "ask": it.get("ask", ""),
                           "fallback": it.get("fallback", ""), "grounded": "", "specific": "", "actionable": ""}
                          for cid, items in plans.items() for it in items])
    sheet.to_csv(OUT / "ratings_template.csv", index=False)
    print(len(sheet), "plan items to rate ->", OUT / "ratings_template.csv")
    plan_judge = pd.DataFrame([{"contract_id": r.contract_id, "id": r.id,
                                **judge(PLAN_JUDGE.format(family=r.family, verdict="", severity=r.severity,
                                                          quote=r.evidence, ask=r.ask, fallback=r.fallback))}
                               for r in sheet.itertuples()]).set_index(["contract_id", "id"])
    if "grounded" in plan_judge:
        save(plan_judge, "llm_judge_plan")
        print("judge means:", plan_judge[["grounded", "specific", "actionable"]].mean().round(2).to_dict())

In [ ]:
CRITERIA = ["grounded", "specific", "actionable"]
rated = {p.stem.split("_", 1)[1]: pd.read_csv(p).set_index(["contract_id", "id"])
         for p in sorted(Path(".").glob("ratings_*.csv")) if p.stem != "ratings_template"}
if rated and "plan_judge" in globals() and "grounded" in plan_judge:
    rated["llm_judge"] = plan_judge


def wkappa(x, y):
    both = x.join(y, lsuffix="_a", rsuffix="_b", how="inner").dropna()
    return {k: cohen_kappa_score(both[k + "_a"].astype(int), both[k + "_b"].astype(int), weights="quadratic")
            for k in CRITERIA}


if rated:
    display(save(pd.DataFrame({who: r[CRITERIA].mean() for who, r in rated.items()}).round(2), "track_c_means"))
    names = list(rated)
    pairs = {f"{a} vs {b}": wkappa(rated[a][CRITERIA], rated[b][CRITERIA])
             for i, a in enumerate(names) for b in names[i + 1:]}
    display(save(pd.DataFrame(pairs).T.round(2), "track_c_agreement"))   # judge vs human = can we trust the judge
else:
    print("no ratings_*.csv yet")

## 7. Save

Everything in `scores/` goes into the submission ZIP and the report.

In [ ]:
print(sorted(p.name for p in OUT.iterdir()))
if IN_COLAB:
    !zip -qr scores.zip scores
    files.download("scores.zip")